# Note that this project is coded for R

![NYC Skyline](nyc.jpg)

Welcome to New York City, one of the most-visited cities in the world. There are many Airbnb listings in New York City to meet the high demand for temporary lodging for travelers, which can be anywhere between a few nights to many months. In this project, you will take a closer look at the New York Airbnb market by combining data from multiple file types like `.csv`, `.tsv`, and `.xlsx` (Excel files).

Recall that **CSV**, **TSV**, and **Excel** files are three common formats for storing data. 
Three files containing data on 2019 Airbnb listings are available to you:

**data/airbnb_price.csv**
This is a CSV file containing data on Airbnb listing prices and locations.
- **`listing_id`**: unique identifier of listing
- **`price`**: nightly listing price in USD
- **`nbhood_full`**: name of borough and neighborhood where listing is located

**data/airbnb_room_type.xlsx**
This is an Excel file containing data on Airbnb listing descriptions and room types.
- **`listing_id`**: unique identifier of listing
- **`description`**: listing description
- **`room_type`**: Airbnb has three types of rooms: shared rooms, private rooms, and entire homes/apartments

**data/airbnb_last_review.tsv**
This is a TSV file containing data on Airbnb host names and review dates.
- **`listing_id`**: unique identifier of listing
- **`host_name`**: name of listing host
- **`last_review`**: date when the listing was last reviewed


In [1]:
# We've loaded the necessary packages for you in the first cell. Please feel free to add as many cells as you like!
suppressMessages(library(dplyr)) # This line is required to check your answer correctly
options(readr.show_types = FALSE) # This line is required to check your answer correctly
library(readr)
library(readxl)
library(stringr)

library(dplyr)
library(lubridate)

# Objectives

As a consultant working for a real estate start-up, you have collected Airbnb listing data from various sources to investigate the short-term rental market in New York. You'll analyze this data to provide insights on private rooms to the real estate company.

There are three files in the data folder: `airbnb_price.csv`, `airbnb_room_type.xlsx`, `airbnb_last_review.tsv`.
- What are the dates of the earliest and most recent reviews?
- How many of the listings are private rooms?
- What is the average listing price for all rooms (rounded to the nearest penny)?
    - Combine these four values into one tibble called `review_dates` with four columns: `first_reviewed`, `last_reviewed`, `nb_private_rooms`, and `avg_price`. The tibble should only contain one row of values.

In [ ]:
#Firstly, need to import the 3 datasets. In this case, the datasets were not in another folder. If they were, 
    #use the following code: read.csv("folder/file")

#To view functions in a package:
    #lsf.str("package:readr")

price_data = read.csv("airbnb_price.csv")
    # 25,209 rows
    # "listing_id": int, "price": chr, "nbhood_full": chr
room_data = read_excel("airbnb_room_type.xlsx")
    # 25,209 rows
    # "listing_id": dbl, "description": chr, "room_type": chr
review_data = read_tsv("airbnb_last_review.tsv")
    # 25,209 rows
    # "listing_id": dbl, "host_name": chr, "last_review": chr

print(price_data)
#Can check data types of columns using glimpse()

In [ ]:
#Next, inspect the data to check for appropriate data types, data inconsistencies, missing values, duplicates

#inspect by data frame
price_data %>%
    count(listing_id) %>%
    arrange(desc(n))

sum(is.na(price_data))


### Observations while inspecting the data frames.
**price_data:**
- There don't appear to be any typos, inconsistencies. There are no missing values. There are no full duplicates.
- In order to do arithmetic on the `price` variable, it needs to be converted from a "character" to a "numeric" type. This involves removing the "dollars" string from this column.

**room_data:**
- There are case inconsistencies in the `room_type` column that need to be fixed (e.g. "shared room," "SHARED ROOM").
- In the `description` column, there are some inconsistencies, but there are so many unique values with little to no structure which makes it difficult to standardize this column.
- There are ten missing values, all of which are in the `description` column. In all of these associated observations, the `room_type` is a "private room." There are so many unique `description`s, so imputing for these missing instances, particularly given that there are no duplicate `listing_id`s, is challenging.
- There are no full duplicates.

**review_data:**
- In the `host_name` column, there are some inconsistencies such as:
    - Case inconsistencies,
    - Sometimes only an initial is provided,
    - Sometimes multiple people are listed with inconsistent conjunctions ("&", "and", "or", "/"),
    - Some non-names (e.g. "Newyorkroomwithaview")
- The `last_review` column is originally a "character" type, but it would be better if it were a "Date" type. Fortunately, the values in this column are formatted consistently & aren't out of range.
- There are eight missing values, all of which are in the `host_name` column. Similarly to "room_data," imputing for these missing values given the associated values would be challenging.
- There are no full duplicates.

In summary, the most pressing issues following these observations include:
- Case inconsistencies in the `room_type` column of "room_data."
- Missing values in "room_data" & "review_data." It is not immediately evident as to how these values should be handled. Fortunately, they associate with a small enough subset of the data sets that they could be dropped.
- Convert dates in `last_review` of "review_data" to "Date" types.

In [ ]:
#Clean data as necessary

#Cross reference "room_data" & "review_data" to see if any of the observations with missing values are related
    # --> they are not
room_data %>%
    filter(is.na(description))
review_data %>%
    filter(is.na(host_name))

#Fix case inconsistencies in 'room_type' of "room_data"
room_data$room_type <- str_to_sentence(room_data$room_type)

#Convert 'last_review' from "chr" to "Date"s
review_data$last_review <- as.Date(review_data$last_review, format="%B %d %Y")
class(review_data$last_review)
    # --> "Date" -- GOOD

#Remove the "dollars" string from 'price' & convert it from "chr" to "numeric"
price_data$price <- as.numeric(str_remove_all(price_data$price, "dollars"))


In [ ]:
#Next, can merge all data together using the "listing_id" variable

merged_data <- full_join(full_join(price_data, room_data, by="listing_id"), review_data, by="listing_id")
merged_data

# Q1
What are the dates of the earliest and most recent reviews?

In [ ]:
first_reviewed <- min(merged_data$last_review)
    # --> 2019-01-01
last_reviewed <- max(merged_data$last_review)
    # --> 2019-07-09

# Q2
How many of the listings are private rooms?

In [ ]:
nb_private_rooms <- count(filter(merged_data, room_type == "Private room"))
    # 11,356

# Q3
What is the average listing price for all rooms (rounded to the nearest penny)?

In [ ]:
avg_price <- round(mean(merged_data$price), digits=2)
    # 141.78

# Compile results
Combine these four values (across Q1, Q2, Q3) into one tibble called review_dates with four columns: first_reviewed, last_reviewed, nb_private_rooms, and avg_price. The tibble should only contain one row of values.

In [ ]:
#Construct tibble
review_dates <- tibble(first_reviewed=first_reviewed, last_reviewed=last_reviewed, nb_private_rooms=nb_private_rooms, avg_price=avg_price)
review_dates
